# Tarea 5 IA Generacion de imagenes con autoencoder
### Javier Alonso Rojas Rojas, Brandon Emmanuel Sanchez Araya, Julio Josue Varela Venegas

In [14]:
import os
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader
import lightning as L
import hydra

In [31]:
import torch.nn as nn

class ClassicAE(nn.Module):
    def __init__(self, latent_dim=128, **kwargs):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),   # (B, 32, 64, 64)
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # (B, 64, 32, 32)
            nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), # (B, 128, 16, 16)
            nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),# (B, 256, 8, 8)
            nn.ReLU(),
        )
        self.flatten = nn.Flatten()  # output = 256*8*8 = 16384
        self.fc_mu = nn.Linear(256 * 8 * 8, latent_dim)


        self.fc_decode = nn.Linear(latent_dim, 256 * 8 * 8)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), # 16x16
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # 32x32
            nn.ReLU(),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),   # 64x64
            nn.ReLU(),

            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),    # 128x128
            nn.Sigmoid()  # imágen RGB normalizada 0–1
        )

    def forward(self, x):
        # Encode
        z = self.encoder(x)
        z = self.flatten(z)
        z = self.fc_mu(z)

        # Decode
        out = self.fc_decode(z)
        out = out.view(-1, 256, 8, 8)
        out = self.decoder(out)

        return out, z

In [32]:
import torch
import torch.nn as nn

class UNetAE(nn.Module):
    def __init__(self, latent_dim=128, **kwargs):
        super().__init__()
        
        # ============================================
        # ENCODER con capas individuales para skip connections
        # ============================================
        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),   # (B, 32, 64, 64)
            nn.ReLU()
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # (B, 64, 32, 32)
            nn.ReLU()
        )
        self.enc3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), # (B, 128, 16, 16)
            nn.ReLU()
        )
        self.enc4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),# (B, 256, 8, 8)
            nn.ReLU()
        )
        
        # ============================================
        # BOTTLENECK (Latent Space)
        # ============================================
        self.flatten = nn.Flatten()  # 256*8*8 = 16384
        self.fc_mu = nn.Linear(256 * 8 * 8, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 256 * 8 * 8)
        
        # ============================================
        # DECODER con skip connections
        # ============================================
        # Nota: Los canales de entrada se duplican por las skip connections
        
        self.dec1 = nn.Sequential(
            # 256 (from bottleneck) + 256 (skip from enc4) = 512 input channels
            nn.ConvTranspose2d(256 + 256, 128, kernel_size=4, stride=2, padding=1), # 16x16
            nn.ReLU()
        )
        self.dec2 = nn.Sequential(
            # 128 (from dec1) + 128 (skip from enc3) = 256 input channels
            nn.ConvTranspose2d(128 + 128, 64, kernel_size=4, stride=2, padding=1),  # 32x32
            nn.ReLU()
        )
        self.dec3 = nn.Sequential(
            # 64 (from dec2) + 64 (skip from enc2) = 128 input channels
            nn.ConvTranspose2d(64 + 64, 32, kernel_size=4, stride=2, padding=1),   # 64x64
            nn.ReLU()
        )
        self.dec4 = nn.Sequential(
            # 32 (from dec3) + 32 (skip from enc1) = 64 input channels
            nn.ConvTranspose2d(32 + 32, 3, kernel_size=4, stride=2, padding=1),    # 128x128
            nn.Sigmoid()  # imagen RGB normalizada 0-1
        )
    
    def forward(self, x):
        # ============================================
        # ENCODER - Guardar skip connections
        # ============================================
        skip1 = self.enc1(x)       # (B, 32, 64, 64)
        skip2 = self.enc2(skip1)   # (B, 64, 32, 32)
        skip3 = self.enc3(skip2)   # (B, 128, 16, 16)
        skip4 = self.enc4(skip3)   # (B, 256, 8, 8)
        
        # ============================================
        # BOTTLENECK
        # ============================================
        z = self.flatten(skip4)
        z = self.fc_mu(z)
        
        # ============================================
        # DECODER - Usar skip connections
        # ============================================
        out = self.fc_decode(z)
        out = out.view(-1, 256, 8, 8)
        
        # Concatenar skip connection 4
        out = torch.cat([out, skip4], dim=1)  # (B, 512, 8, 8)
        out = self.dec1(out)                   # (B, 128, 16, 16)
        
        # Concatenar skip connection 3
        out = torch.cat([out, skip3], dim=1)  # (B, 256, 16, 16)
        out = self.dec2(out)                   # (B, 64, 32, 32)
        
        # Concatenar skip connection 2
        out = torch.cat([out, skip2], dim=1)  # (B, 128, 32, 32)
        out = self.dec3(out)                   # (B, 32, 64, 64)
        
        # Concatenar skip connection 1
        out = torch.cat([out, skip1], dim=1)  # (B, 64, 64, 64)
        out = self.dec4(out)                   # (B, 3, 128, 128)
        
        return out, z

In [33]:
class LitAutoencoder(L.LightningModule):
    def __init__(self, cfg):
        super().__init__()
        # Hydra instancia automáticamente el modelo desde cfg.model
        self.model = hydra.utils.instantiate(cfg.model)
        
        # Para la loss function, también usa instantiate
        self.loss_fn = hydra.utils.instantiate(cfg.loss)
        
        self.lr = cfg.trainer.learning_rate

    def training_step(self, batch, _):
        x = batch
        x_rec, z = self.model(x)
        loss = self.loss_fn(x_rec, x)
        self.log("train_loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

## Configuración con Hydra

Ahora puedes cambiar entre experimentos fácilmente:

In [42]:
from hydra import initialize_config_dir, compose
from pathlib import Path
import hydra

# Inicializar Hydra
config_dir = Path("conf").absolute()
print(f"Usando directorio de configuración: {config_dir}")

with initialize_config_dir(config_dir=str(config_dir), version_base=None):
    # Cambiar entre experimentos:
    # - classic_l1, classic_l2, classic_ssim, classic_ssim_l1
    # - unet_l1, unet_l2, unet_ssim, unet_ssim_l1
    
    cfg = compose(config_name="config")
    
    print(f"Experimento: {cfg.experiment_name}")
    print(f"Modelo: {cfg.model.name}")
    print(f"Arquitectura: {cfg.model.architecture}")
    print(f"Loss: {cfg.loss._target_}")
    print(f"Latent dim: {cfg.model.latent_dim}")
    print(f"Learning rate: {cfg.trainer.learning_rate}")
    print(f"Max epochs: {cfg.trainer.max_epochs}")
    
    # Crear el modelo Lightning
    lit_model = LitAutoencoder(cfg)
    print(f"\nModelo creado: {lit_model.model.__class__.__name__}")

Usando directorio de configuración: /home/javialroro/TEC/2025 II/IA/AutoencoderImageGen/conf
Experimento: classic_ae_l2
Modelo: classic_autoencoder
Arquitectura: classic
Loss: torch.nn.MSELoss
Latent dim: 128
Learning rate: 0.001
Max epochs: 50

Modelo creado: ClassicAE
